# Lab | BabyAGI with agent

**Change the planner objective below by changing the objective and the associated prompts and potential tools and agents - Wear your creativity and AI engineering hats
You can't get this wrong!**

You would need the OpenAI API KEY and the [SerpAPI KEY](https://serpapi.com/manage-api-keyhttps://serpapi.com/manage-api-key) to run this lab.


## BabyAGI with Tools

This notebook builds on top of [baby agi](baby_agi.html), but shows how you can swap out the execution chain. The previous execution chain was just an LLM which made stuff up. By swapping it out with an agent that has access to tools, we can hopefully get real reliable information

## Install and Import Required Modules

In [18]:
!pip install langchain langchain-community langchain-experimental langchain-openai langchain-classic


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
from typing import Optional

# Legacy chains/prompts → langchain-classic
from langchain_classic.chains import LLMChain
from langchain_classic.prompts import PromptTemplate

# BabyAGI still in langchain-experimental
from langchain_experimental.autonomous_agents import BabyAGI

# OpenAI integrations
from langchain_openai import OpenAI, OpenAIEmbeddings

## Connect to the Vector Store

Depending on what vectorstore you use, this step may look different.

In [20]:
# # %pip install faiss-cpu > /dev/null
# # %pip install google-search-results > /dev/null
# from langchain.docstore import InMemoryDocstore
# from langchain_community.vectorstores import FAISS

In [21]:
!pip install faiss-cpu langchain-community


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore  # Updated path
from langchain_community.vectorstores import FAISS

In [23]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
SERPAPI_API_KEY = os.getenv('SERPAPI_API_KEY')

In [24]:
# Define your embedding model
from langchain_openai import OpenAIEmbeddings
embeddings_model = OpenAIEmbeddings()

# Initialize the vectorstore as empty
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

embedding_size = 1536
index = faiss.IndexFlatL2(embedding_size)
vectorstore = FAISS(
    embedding_function=embeddings_model,  # Full embeddings object, not .embed_query
    index=index,
    docstore=InMemoryDocstore({}),
    index_to_docstore_id={}
)

## Define the Chains

BabyAGI relies on three LLM chains:
- Task creation chain to select new tasks to add to the list
- Task prioritization chain to re-prioritize tasks
- Execution Chain to execute the tasks


NOTE: in this notebook, the Execution chain will now be an agent.

In [25]:
!pip install google-search-results


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
from langchain_classic.agents import AgentExecutor, Tool, ZeroShotAgent  # Legacy agents
from langchain_classic.chains import LLMChain  # Legacy chains
from langchain_classic.prompts import PromptTemplate  # Legacy prompts
from langchain_community.utilities import SerpAPIWrapper
from langchain_openai import OpenAI

todo_prompt = PromptTemplate.from_template(
    """
You are a trip-architect planner. Produce a TODO plan to achieve this objective:

OBJECTIVE:
{objective}

Requirements:
- Break into 8–14 atomic tasks
- Each task must be directly actionable and verifiable
- Include: budget validation, opening-hours verification, route/transit validation, and 2 indoor backup options
- Explicitly call out constraints you must not violate (budget cap, timing, interests)
- Output as a numbered checklist with short task titles + 1 sentence detail each
"""
)
# --- Itinerary formatter chain (THIS was missing) ---
itinerary_prompt = PromptTemplate.from_template(
    """
You are an itinerary formatter.

Input notes:
{notes}

Produce:
- A timeboxed schedule (start–end times)
- For each stop: address (if known), why it fits, approximate cost, and transit step (U/S-bahn/bus/walk)
- Total estimated cost at the end
- Two indoor backup options (with where to swap them in)

Keep it concise and practical.
"""
)
itinerary_chain = LLMChain(llm=OpenAI(temperature=0), prompt=todo_prompt)
, prompt=itinerary_prompt)
search = SerpAPIWrapper()
def budget_check(expr: str) -> str:
    expr = expr.strip().strip("'").strip('"')
    try:
        allowed = set("0123456789+-(). ")
        if any(ch not in allowed for ch in expr):
            return "BudgetCheckError: Invalid characters."
        total = eval(expr, {"__builtins__": {}}, {})
        return f"Total = {total:.2f}"
    except Exception as e:
        return f"BudgetCheckError: {e}"

tools = [
    Tool(
        name="Search",
        func=search.run,
        description="Use to verify opening hours, ticket prices, transit changes, and current info for locations."
    ),
    Tool(
        name="TODO",
        func=todo_chain.run,
        description="Use to generate a structured checklist for the itinerary objective. Input: objective. Output: numbered TODO plan."
    ),
    Tool(
        name="BUDGET_CHECK",
        func=budget_check,
        description="Use to add up estimated costs. Input: an arithmetic expression like '12 + 18 + 9.5'. Output: total."
    ),
    Tool(
        name="ITINERARY_FORMATTER",
        func=itinerary_chain.run,
        description="Use to convert notes into a clean timeboxed itinerary. Input: notes. Output: formatted schedule."
    ),
]


prefix = """You are a constraint-driven Trip Architect.
You must perform ONE task based on the objective: {objective}.

Use tools to:
- verify prices/opening hours/transit
- keep the plan under the stated budget
- provide indoor backup options

Previously completed tasks (do not repeat them): {context}

If information is missing, make a reasonable assumption AND clearly label it as an assumption.
"""

suffix = """Question: {task}
{agent_scratchpad}"""
prompt = ZeroShotAgent.create_prompt(
    tools,
    prefix=prefix,
    suffix=suffix,
    input_variables=["objective", "task", "context", "agent_scratchpad"],
)

In [27]:
llm = OpenAI(temperature=0)
llm_chain = LLMChain(llm=llm, prompt=prompt)
tool_names = [tool.name for tool in tools]
agent = ZeroShotAgent(llm_chain=llm_chain, allowed_tools=tool_names)
agent_executor = AgentExecutor.from_agent_and_tools(
    agent=agent, tools=tools, verbose=True,max_iterations=2,
    early_stopping_method="force"
)

### Run the BabyAGI

Now it's time to create the BabyAGI controller and watch it try to accomplish your objective.

In [28]:
OBJECTIVE = "Design a 1-day Berlin itinerary for a foodie + modern art lover under €120, including 2 indoor backups and exact transit notes."


In [29]:
from typing import Optional
from langchain_experimental.autonomous_agents import BabyAGI

In [30]:
# Logging of LLMChains
verbose = False
# If None, will keep on going forever
max_iterations: Optional[int] = 1
baby_agi = BabyAGI.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    task_execution_chain=agent_executor,
    verbose=verbose,
    max_iterations=max_iterations,
)

In [31]:
baby_agi({"objective": OBJECTIVE})


*****TASK LIST*****

1: Make a todo list

*****NEXT TASK*****

1: Make a todo list


> Entering new AgentExecutor chain...
Thought: I should use the TODO function to generate a structured checklist
Action: TODO
Action Input: Design a 1-day Berlin itinerary for a foodie + modern art lover
Observation: 
1. Research and select top modern art museums in Berlin within budget constraints.
2. Verify opening hours and admission prices for selected museums.
3. Plan route and transit options to visit each museum efficiently.
4. Budget validation: Ensure total cost of museum admissions and transportation stays within budget cap.
5. Research and select top foodie spots in Berlin within budget constraints.
6. Verify opening hours and menu options for selected foodie spots.
7. Plan route and transit options to visit each foodie spot efficiently.
8. Budget validation: Ensure total cost of meals stays within budget cap.
9. Create a schedule for the day, taking into account opening hours and transit t

{'objective': 'Design a 1-day Berlin itinerary for a foodie + modern art lover under €120, including 2 indoor backups and exact transit notes.'}